# 01 — Recorded defense demo: Flip/ReFlip on Llama-3-8B

**Attach before running:** the two datasets built by notebook `00`
(`llama3-rtn-demo`, `llama3-rtn-reflip-demo`) and the `HF_TOKEN` secret.
Accelerator: **GPU T4 x2** - Internet **On**.

Timeline (record the screen, cut the eval waits in editing):
1. **Demo 1** - standalone Q-K case study (layer 8, GQA group 3): error table + Kneedle figures.
2. **Demo 2** - full precision vs RTN vs RTN+ReFlip: WikiText perplexity + ARC-Easy subset,
   then the concrete questions ReFlip fixed.


In [ ]:
# --- Setup: repo, deps, checkpoint paths ---
import glob, os, subprocess, sys

if not os.path.exists("/kaggle/tmp/repo"):
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Itvc0110/Reflip-Flip-on-QKV-.git", "/kaggle/tmp/repo"], check=True)
os.chdir("/kaggle/tmp/repo")

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers>=4.45,<4.53", "datasets>=2.20,<3.0", "accelerate",
                "sentencepiece", "kneed", "matplotlib", "pandas",
                "lm-eval>=0.4,<0.5"], check=True)

def find_input(pattern):
    hits = glob.glob(f"/kaggle/input/*/{pattern}") + glob.glob(f"/kaggle/input/{pattern}")
    assert hits, f"attach the dataset containing '{pattern}'"
    return hits[0]

RTN_PATH    = find_input("llama3_rtn")
REFLIP_PATH = find_input("llama3_rtn_reflip")
DEMO1_DIR   = find_input("demo1")
print("RTN checkpoint:   ", RTN_PATH)
print("ReFlip checkpoint:", REFLIP_PATH)
print("Demo-1 artifacts: ", DEMO1_DIR)

## Demo 1 — Standalone Q–K case study (Llama-3-8B, layer 8, GQA group 3)

One GQA group under the microscope: 4 query heads sharing one key head, asymmetric INT4
(group size 128). Three successive states — **RTN → Flip → Flip + ReFlip** — evaluated on the
exact scalar Q–K surrogate each stage optimizes. Artifacts were produced by
`xspot.py` → `fast_quantize_qkv.py` (same pipeline as the thesis).

In [ ]:
# --- Demo 1: per-head error table (matches thesis Tables 4.2/4.3) ---
import subprocess, sys
subprocess.run([sys.executable, "tools/summarize_qkv_results.py",
                "--npz", f"{DEMO1_DIR}/quantization_results.npz"], check=True)

In [ ]:
# --- Demo 1: figures (regenerated live from the run's npz) ---
import subprocess, sys
from IPython.display import Image, display

subprocess.run([sys.executable, "tools/plot_kneedle_sensitivity.py",
                "--npz", f"{DEMO1_DIR}/quantization_results.npz",
                "--out", "/kaggle/working/reflip_kneedle.png"], check=True)
subprocess.run([sys.executable, "tools/plot_flip_activation_kneedle.py",
                "--npz", f"{DEMO1_DIR}/quantization_results.npz",
                "--out", "/kaggle/working/flip_kneedle.png"], check=True)

display(Image("/kaggle/working/flip_kneedle.png", width=760))
display(Image("/kaggle/working/reflip_kneedle.png", width=760))
display(Image(f"{DEMO1_DIR}/attention_quantization_analysis.png", width=860))

## Demo 2 — Full model: full precision vs RTN vs RTN + ReFlip

Each variant is evaluated **sequentially** (one model in GPU memory at a time;
`parallelize=True` shards the 8B model across both T4s). WikiText word-perplexity +
ARC-Easy (250-question subset, seeded) via the `lm-eval` harness — the same
log-likelihood decision rule as the thesis benchmarks.

In [ ]:
# --- Download the full-precision reference (scratch dir, ~16 GB) ---
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login, snapshot_download

login(token=UserSecretsClient().get_secret("HF_TOKEN"))
FP_PATH = "/kaggle/tmp/models/Llama-3-8B"
snapshot_download("meta-llama/Meta-Llama-3-8B", local_dir=FP_PATH,
                  ignore_patterns=["original/*", "*.pth"])
print("full-precision model at", FP_PATH)

In [ ]:
# --- Evaluate all three variants (the long cell: ~35-45 min total on T4 x2) ---
import subprocess, sys

VARIANTS = [
    ("full_precision",     FP_PATH),
    ("llama3_rtn",         RTN_PATH),
    ("llama3_rtn_reflip",  REFLIP_PATH),
]

for label, path in VARIANTS:
    print("\n" + "=" * 70)
    print("Evaluating:", label)
    print("=" * 70)
    subprocess.run([
        sys.executable, "-m", "lm_eval",
        "--model", "hf",
        "--model_args", f"pretrained={path},dtype=float16,parallelize=True",
        "--tasks", "wikitext,arc_easy",
        "--limit", "250",
        "--batch_size", "4",
        "--seed", "42",
        "--log_samples",
        "--output_path", f"/kaggle/working/lm_eval/{label}",
    ], check=True)   # subprocess exit frees the GPU before the next variant

In [ ]:
# --- Results table ---
import glob, json
import pandas as pd

rows = []
for label, _ in VARIANTS:
    result_files = sorted(glob.glob(f"/kaggle/working/lm_eval/{label}/**/results_*.json",
                                    recursive=True))
    res = json.load(open(result_files[-1]))["results"]
    rows.append({
        "model": label,
        "wikitext word_ppl (lower=better)": round(res["wikitext"]["word_perplexity,none"], 4),
        "arc_easy acc (250 q, higher=better)": round(res["arc_easy"]["acc,none"], 4),
    })
df = pd.DataFrame(rows).set_index("model")
display(df)
print("\nThesis Table 4.7 (full ARC-Easy, full sliding-window ppl) for direction reference:")
print("  RTN: wiki 6.0572, arc-e 0.780   |   RTN + ReFlip: wiki 6.0236, arc-e 0.781")
print("  five-task avg: 0.6924 -> 0.7033 (+1.09 points)")

In [ ]:
# --- The questions ReFlip fixed (RTN wrong -> RTN+ReFlip right) ---
import subprocess, sys, glob
from IPython.display import Image, display

subprocess.run([sys.executable, "tools/compare_lm_eval_samples.py",
                "--baseline-dir", "/kaggle/working/lm_eval/llama3_rtn",
                "--variant-dir",  "/kaggle/working/lm_eval/llama3_rtn_reflip",
                "--task", "arc_easy", "--top-n", "5",
                "--plot-dir", "/kaggle/working/lm_eval/flipped_examples"], check=True)

for png in sorted(glob.glob("/kaggle/working/lm_eval/flipped_examples/*.png"))[:3]:
    display(Image(png, width=780))

## Wrap-up

- **Demo 1**: on the exact scalar Q–K surrogate, mean abs error fell
  0.1428 → 0.0701 (Flip, −50.9%) → 0.0479 (Flip + ReFlip, −66.4%) while changing only
  ~0.8% of the weights + 64 extra one-level moves — the numbers on screen match the thesis.
- **Demo 2**: the same correction, applied to the whole model, moves perplexity down and
  ARC-Easy accuracy up relative to RTN — the same direction as thesis Table 4.7 — and the
  flipped-question examples show *where* those extra points come from.

*(This is a fresh run: 250-question subset + lm-eval's word-perplexity protocol; presented as
directionally consistent with the thesis tables, not a bit-exact reproduction.)*